# Run New Signals: Team Polarisation + Team Centroid Distance

This notebook processes one or more matches through the two new signals
(`team_polarisation` and `team_centroid_distance`) and then merges all
signal outputs into the unified dataset.

**Author:** Conor Malone
**Date:** July 2026

---

## Pre-requisites

- Tracking data in Parquet format (StatsPerform data)
- Shape model JSON files (for positional drift — not needed here)
- Python environment with: pandas, numpy, matplotlib, scipy

In [ ]:
# ── Cell 1: Imports & Path Setup ───────────────────────────────────────

import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().resolve().parent  # notebooks/ -> project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

In [ ]:
# ── Cell 2: Import Signal Modules (triggers @register_signal decorators) ─

import pandas as pd
import numpy as np

# Import the new signal modules — these must come before list_signals
import src.signals.polarisation           # noqa: F401
import src.signals.team_centroid_distance  # noqa: F401

# Pipeline helpers
from src.loaders import load_tracking_statsperform
from src.segments import split_into_blocks
from src.smoothing import smooth_trajectory, compute_velocity_features
from src.signals.registry import list_signals, SIGNAL_REGISTRY

# Config
from src.pressure.config import DEFAULT_CONFIG as PRESSURE_CONFIG

print(f"Registered signals ({len(list_signals())}): {list_signals()}")
assert 'team_polarisation' in list_signals(), "team_polarisation not registered!"
assert 'team_centroid_distance' in list_signals(), "team_centroid_distance not registered!"
print("✅ Both new signals registered successfully.")

---

## Helper: Process One Match

Loads tracking data, computes smoothing + velocity features, splits into
5-minute blocks, then runs **only** the two new signals.

In [ ]:
# ── Cell 3: Process a Single Match Through New Signals ────────────────

NEW_SIGNALS = ['team_polarisation', 'team_centroid_distance']


def process_match_new_signals(
    match_id: str,
    tracking_path: str,
    config=None,
    nrows: int | None = None,
) -> dict:
    """Run the two new signals on a single match.

    Parameters
    ----------
    match_id : str
        Match identifier.
    tracking_path : str
        Path to the tracking.parquet file.
    config : PressureConfig, optional
        Pressure config for block window settings.
    nrows : int, optional
        Limit rows for testing.

    Returns
    -------
    dict
        Summary with match_id and per-signal results.
    """
    config = config or PRESSURE_CONFIG

    # 1. Load tracking data
    df = load_tracking_statsperform(
        str(tracking_path), match_id=match_id,
        normalise_dop=True, include_ball=True
    )
    if nrows:
        df = df.head(nrows)
    df["frame"] = df["frame_count"]
    print(f"  Loaded {len(df):,} rows")

    # 2. Smooth and compute velocities
    df = smooth_trajectory(df, inplace=False)
    compute_velocity_features(df, inplace=True)
    print("  Smoothed + computed velocities")

    # 3. Split into blocks
    blocks = split_into_blocks(
        df, window_minutes=config.block_window_minutes,
        min_frames=config.block_min_frames
    )
    print(f"  {len(blocks)} blocks")

    if len(blocks) == 0:
        print("  ⚠️  No valid blocks!")
        return {"match_id": match_id, "signals": {}}

    # 4. Run each new signal
    results = {}
    for signal_name in NEW_SIGNALS:
        sig_cls = SIGNAL_REGISTRY.get(signal_name)
        if sig_cls is None:
            print(f"  ⚠️  Signal {signal_name} not found")
            results[signal_name] = {"rows": 0, "error": "Not registered"}
            continue

        print(f"\n  Computing {signal_name}...")
        try:
            signal = sig_cls()
            output_df = signal.compute(
                match_df=df, blocks=blocks, game_id=match_id
            )
            signal.save(output_df, match_id=match_id)
            n_rows = len(output_df)
            print(f"  ✅ {n_rows} rows saved")
            results[signal_name] = {"rows": n_rows}
        except Exception as e:
            print(f"  ❌ Error: {e}")
            results[signal_name] = {"rows": 0, "error": str(e)}

    return {"match_id": match_id, "signals": results}

---

## Run on Available Sample Matches

In [ ]:
# ── Cell 4: Discover matches and run ──────────────────────────────────

SAMPLE_DIR = Path("./data/sample")
TRACKING_DIR = Path("./data/raw/tracking")

def find_available_matches():
    """Find matches with tracking.parquet files."""
    matches = []
    for d in [SAMPLE_DIR, TRACKING_DIR]:
        if d.exists():
            for sub in sorted(d.iterdir()):
                if sub.is_dir() and (sub / "tracking.parquet").exists():
                    matches.append((sub.name, sub / "tracking.parquet"))
    return matches

available = find_available_matches()
print(f"Found {len(available)} available matches:")
for mid, _ in available[:5]:
    print(f"  - {mid}")
if len(available) > 5:
    print(f"  ... and {len(available) - 5} more")

In [ ]:
# ── Cell 5: Run new signals on all (or a subset of) matches ───────────

# Set to a subset for quick testing, or run on all
MATCHES_TO_PROCESS = available[:2]  # First 2 matches for testing
# MATCHES_TO_PROCESS = available    # Uncomment for full run

all_results = []
for match_id, tracking_path in MATCHES_TO_PROCESS:
    print(f"\n{'='*60}")
    print(f"Processing: {match_id}")
    print(f"{'='*60}")
    result = process_match_new_signals(
        match_id, tracking_path, nrows=50000  # limit for testing
    )
    all_results.append(result)

print(f"\n✅ Completed {len(all_results)} matches")

---

## Summary

In [ ]:
# ── Cell 6: Print summary of results ──────────────────────────────────

print(f"{'Signal':<30} {'Total Rows':>12}  {'Matches':>8}  {'Errors':>8}")
print(f"{'-'*62}")
for signal_name in NEW_SIGNALS:
    total_rows = sum(
        r["signals"].get(signal_name, {}).get("rows", 0)
        for r in all_results
    )
    n_errors = sum(
        1 for r in all_results
        if r["signals"].get(signal_name, {}).get("error")
    )
    n_matches = sum(
        1 for r in all_results
        if r["signals"].get(signal_name, {})
    )
    print(f"{signal_name:<30} {total_rows:>12,}  {n_matches:>8}  {n_errors:>8}")

---

## Merge All Signal Outputs

After signals are computed for all matches, merge into the unified dataset.

In [ ]:
# ── Cell 7: Merge all signal outputs ──────────────────────────────────

# Optional: verify the new signal files exist before merging
from pathlib import Path

OUTPUT_ROOT = Path("outputs/signals")
print("Checking signal output directories...")
for sig_name in NEW_SIGNALS:
    sig_dir = OUTPUT_ROOT / sig_name
    if sig_dir.exists():
        files = list(sig_dir.glob("*.csv"))
        print(f"  {sig_name}: {len(files)} CSV files in {sig_dir}")
    else:
        print(f"  {sig_name}: directory does not exist")

# Run merge
from src.merge_outputs import merge_all

print("\nMerging all signal outputs...")
try:
    merge_all(output_path="./outputs/unified_fatigue_dataset.parquet")
    print("✅ Merge complete.")
except Exception as e:
    print(f"❌ Merge failed: {e}")

In [ ]:
# ── Cell 8 (Optional): Quick visualisation of new signals ─────────────

import matplotlib.pyplot as plt
import seaborn as sns

# Load merged dataset
merged_path = Path("outputs/unified_fatigue_dataset.parquet")
if merged_path.exists():
    merged = pd.read_parquet(merged_path)
    
    # Filter to new signals
    new_sig_data = merged[merged['signal_name'].isin(NEW_SIGNALS)]
    print(f"Unified dataset has {len(merged):,} rows")
    print(f"New signals: {len(new_sig_data):,} rows")
    
    # Quick boxplot of block-by-block values
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, sig_name in zip(axes, NEW_SIGNALS):
        sig_df = new_sig_data[new_sig_data['signal_name'] == sig_name]
        if len(sig_df) > 0:
            ax.boxplot(sig_df['signal_value'].dropna())
            ax.set_title(sig_name)
            ax.set_ylabel('Signal Value')
    plt.tight_layout()
    plt.show()
else:
    print("Merged dataset not yet available.")